In [1]:
import networkx as nx
from difflib import SequenceMatcher
from collections import defaultdict
import numpy as np
import time
from corefeval import get_metrics
import json

In [2]:
# Load datasets accordingly to the subtask

# Select which file we should be working on
file_to_load = "../../SOMD-2026/data/SOMD 2026/subtask 1/train_data.jsonl"
with open(file_to_load, 'r') as f:
    train_data = [json.loads(l) for l in list(f)]

with open("../data/SOMD 2026/subtask 1/train_labels.json", "r") as f:
    train_labels = json.load(f)

In [4]:



class GraphBasedClustering:
    """
    Graph-based clustering for software mention coreference resolution.
    
    Uses connected components on a similarity graph to enable transitive
    closure of mention similarities.
    """
    
    def __init__(self, mentions_data, similarity_threshold=0.80):
        """
        Initialize graph-based clustering.
        
        Args:
            mentions_data: List of mention dictionaries
            similarity_threshold: Minimum similarity to create edge (0.0 to 1.0)
        """
        self.mentions = mentions_data
        self.mention_dict = {m['mention_id']: m for m in mentions_data}
        self.similarity_threshold = similarity_threshold
        self.graph = None
        self.similarity_cache = {}
    
    def normalize_mention(self, mention_text):
        """Normalize mention text for comparison."""
        return mention_text.lower().strip()
    
    def compute_similarity(self, mention1, mention2):
        """
        Compute similarity between two mentions using fuzzy string matching.
        
        Args:
            mention1: First mention dict
            mention2: Second mention dict
        
        Returns:
            Similarity score between 0.0 and 1.0
        """
        # Check cache first
        cache_key = tuple(sorted([mention1['mention_id'], mention2['mention_id']]))
        if cache_key in self.similarity_cache:
            return self.similarity_cache[cache_key]
        
        # Normalize texts
        text1 = self.normalize_mention(mention1['mention'])
        text2 = self.normalize_mention(mention2['mention'])
        
        # Compute similarity using SequenceMatcher
        similarity = SequenceMatcher(None, text1, text2).ratio()
        
        # Cache result
        self.similarity_cache[cache_key] = similarity
        
        return similarity
    
    def build_similarity_graph(self):
        """
        Build similarity graph where:
        - Nodes = mentions
        - Edges = similarity above threshold
        
        Returns:
            NetworkX Graph object
        """
        print(f"Building similarity graph...")
        print(f"  Nodes: {len(self.mentions)} mentions")
        print(f"  Similarity threshold: {self.similarity_threshold}")
        
        # Create empty graph
        G = nx.Graph()
        
        # Add all mentions as nodes
        for mention in self.mentions:
            G.add_node(
                mention['mention_id'],
                text=mention['mention'],
                data=mention
            )
        
        # Add edges based on similarity
        edge_count = 0
        total_comparisons = len(self.mentions) * (len(self.mentions) - 1) // 2
        
        print(f"  Computing {total_comparisons:,} pairwise similarities...")
        
        start_time = time.time()
        
        for i, mention1 in enumerate(self.mentions):
            for mention2 in self.mentions[i+1:]:
                # Compute similarity
                sim = self.compute_similarity(mention1, mention2)
                
                # Add edge if above threshold
                if sim >= self.similarity_threshold:
                    G.add_edge(
                        mention1['mention_id'],
                        mention2['mention_id'],
                        weight=sim
                    )
                    edge_count += 1
        
        elapsed = time.time() - start_time
        
        print(f"  Added {edge_count:,} edges in {elapsed:.2f}s")
        print(f"  Average degree: {2 * edge_count / len(self.mentions):.2f}")
        
        self.graph = G
        return G
    
    def find_connected_components(self):
        """
        Find connected components in the similarity graph.
        
        Each connected component = one cluster.
        
        Returns:
            List of clusters (each cluster is list of mention_ids)
        """
        if self.graph is None:
            self.build_similarity_graph()
        
        print(f"\nFinding connected components...")
        
        # Find connected components
        components = nx.connected_components(self.graph)
        
        # Convert to list of clusters
        clusters = [list(component) for component in components]
        
        print(f"  Found {len(clusters)} clusters")
        
        # Print cluster size distribution
        cluster_sizes = [len(c) for c in clusters]
        print(f"  Cluster size distribution:")
        print(f"    Min: {min(cluster_sizes)}")
        print(f"    Max: {max(cluster_sizes)}")
        print(f"    Mean: {np.mean(cluster_sizes):.2f}")
        print(f"    Median: {np.median(cluster_sizes):.1f}")
        print(f"    Singletons: {sum(1 for s in cluster_sizes if s == 1)}")
        
        return clusters
    
    def cluster(self):
        """
        Main clustering method.
        
        Returns:
            List of clusters (each cluster is list of mention_ids)
        """
        return self.find_connected_components()
    
    def get_cluster_statistics(self, clusters):
        """
        Compute statistics about clusters.
        
        Args:
            clusters: List of clusters
        
        Returns:
            Dictionary with statistics
        """
        cluster_sizes = [len(c) for c in clusters]
        
        stats = {
            'n_clusters': len(clusters),
            'n_mentions': sum(cluster_sizes),
            'min_size': min(cluster_sizes) if cluster_sizes else 0,
            'max_size': max(cluster_sizes) if cluster_sizes else 0,
            'mean_size': np.mean(cluster_sizes) if cluster_sizes else 0,
            'median_size': np.median(cluster_sizes) if cluster_sizes else 0,
            'n_singletons': sum(1 for s in cluster_sizes if s == 1),
            'singleton_rate': sum(1 for s in cluster_sizes if s == 1) / len(clusters) if clusters else 0,
        }
        
        return stats
    
    def visualize_cluster(self, cluster_mentions, output_file=None):
        """
        Visualize a single cluster as a graph.
        
        Args:
            cluster_mentions: List of mention_ids in cluster
            output_file: Optional path to save figure
        """
        import matplotlib.pyplot as plt
        
        # Create subgraph for this cluster
        subgraph = self.graph.subgraph(cluster_mentions)
        
        # Layout
        pos = nx.spring_layout(subgraph, k=1, iterations=50)
        
        # Draw
        plt.figure(figsize=(12, 8))
        
        # Draw nodes
        nx.draw_networkx_nodes(
            subgraph, pos,
            node_color='lightblue',
            node_size=1000,
            alpha=0.9
        )
        
        # Draw edges with weights
        edges = subgraph.edges()
        weights = [subgraph[u][v]['weight'] for u, v in edges]
        
        nx.draw_networkx_edges(
            subgraph, pos,
            width=[w * 3 for w in weights],  # Thicker = more similar
            alpha=0.5
        )
        
        # Draw labels (mention text)
        labels = {
            mid: self.mention_dict[mid]['mention']
            for mid in cluster_mentions
        }
        nx.draw_networkx_labels(
            subgraph, pos,
            labels,
            font_size=10,
            font_weight='bold'
        )
        
        # Draw edge labels (similarity scores)
        edge_labels = {
            (u, v): f"{subgraph[u][v]['weight']:.2f}"
            for u, v in edges
        }
        nx.draw_networkx_edge_labels(
            subgraph, pos,
            edge_labels,
            font_size=8
        )
        
        plt.title(f"Cluster with {len(cluster_mentions)} mentions", fontsize=14)
        plt.axis('off')
        plt.tight_layout()
        
        if output_file:
            plt.savefig(output_file, dpi=300, bbox_inches='tight')
            print(f"Saved visualization to {output_file}")
        
        plt.show()
    
    def analyze_transitivity(self, clusters):
        """
        Analyze how many mentions are clustered through transitivity.
        
        A mention pair is "transitive" if they're in the same cluster
        but their direct similarity is below threshold.
        
        Args:
            clusters: List of clusters
        
        Returns:
            Dictionary with transitivity statistics
        """
        total_pairs = 0
        direct_pairs = 0  # Pairs with direct edge
        transitive_pairs = 0  # Pairs connected only transitively
        
        for cluster in clusters:
            if len(cluster) < 2:
                continue
            
            # All pairs in cluster
            for i, mid1 in enumerate(cluster):
                for mid2 in cluster[i+1:]:
                    total_pairs += 1
                    
                    # Check if direct edge exists
                    if self.graph.has_edge(mid1, mid2):
                        direct_pairs += 1
                    else:
                        transitive_pairs += 1
        
        return {
            'total_pairs': total_pairs,
            'direct_pairs': direct_pairs,
            'transitive_pairs': transitive_pairs,
            'transitivity_rate': transitive_pairs / total_pairs if total_pairs > 0 else 0
        }
    
    def compare_with_pairwise(self, pairwise_clusters):
        """
        Compare graph-based clusters with pairwise clustering.
        
        Shows benefit of transitive closure.
        
        Args:
            pairwise_clusters: Clusters from pairwise method (e.g., fuzzy matching)
        
        Returns:
            Comparison statistics
        """
        graph_clusters = self.cluster()
        
        return {
            'pairwise_n_clusters': len(pairwise_clusters),
            'graph_n_clusters': len(graph_clusters),
            'cluster_difference': len(pairwise_clusters) - len(graph_clusters),
            'pairwise_mean_size': np.mean([len(c) for c in pairwise_clusters]),
            'graph_mean_size': np.mean([len(c) for c in graph_clusters]),
        }


# ============================================================================
# Usage Examples
# ============================================================================

def example_basic_usage(train_data):
    """Example 1: Basic usage"""
    
    print("="*70)
    print("EXAMPLE 1: BASIC GRAPH-BASED CLUSTERING")
    print("="*70)
    
    # Initialize
    graph_clustering = GraphBasedClustering(
        mentions_data=train_data,
        similarity_threshold=0.80
    )
    
    # Cluster
    clusters = graph_clustering.cluster()
    
    # Get statistics
    stats = graph_clustering.get_cluster_statistics(clusters)
    
    print("\nCluster Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")
    
    return clusters


def example_threshold_comparison(train_data):
    """Example 2: Compare different thresholds"""
    
    print("\n" + "="*70)
    print("EXAMPLE 2: THRESHOLD COMPARISON")
    print("="*70)
    
    thresholds = [0.70, 0.75, 0.80, 0.85, 0.90]
    
    results = []
    
    for threshold in thresholds:
        print(f"\nThreshold: {threshold}")
        
        clusterer = GraphBasedClustering(
            mentions_data=train_data,
            similarity_threshold=threshold
        )
        
        clusters = clusterer.cluster()
        stats = clusterer.get_cluster_statistics(clusters)
        
        results.append({
            'threshold': threshold,
            'n_clusters': stats['n_clusters'],
            'mean_size': stats['mean_size'],
            'n_singletons': stats['n_singletons']
        })
        
        print(f"  Clusters: {stats['n_clusters']}")
        print(f"  Mean size: {stats['mean_size']:.2f}")
        print(f"  Singletons: {stats['n_singletons']}")
    
    return results


def example_transitivity_analysis(train_data):
    """Example 3: Analyze transitivity benefit"""
    
    print("\n" + "="*70)
    print("EXAMPLE 3: TRANSITIVITY ANALYSIS")
    print("="*70)
    
    clusterer = GraphBasedClustering(
        mentions_data=train_data,
        similarity_threshold=0.80
    )
    
    clusters = clusterer.cluster()
    
    # Analyze transitivity
    trans_stats = clusterer.analyze_transitivity(clusters)
    
    print("\nTransitivity Statistics:")
    print(f"  Total mention pairs in clusters: {trans_stats['total_pairs']}")
    print(f"  Direct edges (sim ≥ threshold): {trans_stats['direct_pairs']}")
    print(f"  Transitive connections: {trans_stats['transitive_pairs']}")
    print(f"  Transitivity rate: {trans_stats['transitivity_rate']:.1%}")
    
    print("\nInterpretation:")
    if trans_stats['transitivity_rate'] > 0.05:
        print(f"  → {trans_stats['transitivity_rate']:.1%} of clustered pairs are connected")
        print(f"    only through transitivity (not directly similar).")
        print(f"  → Graph-based approach adds value beyond pairwise matching!")
    else:
        print(f"  → Low transitivity rate suggests graph structure doesn't add much")
        print(f"    over simple pairwise clustering for this data.")


def example_visualization(train_data):
    """Example 4: Visualize a cluster"""
    
    print("\n" + "="*70)
    print("EXAMPLE 4: CLUSTER VISUALIZATION")
    print("="*70)
    
    clusterer = GraphBasedClustering(
        mentions_data=train_data,
        similarity_threshold=0.80
    )
    
    clusters = clusterer.cluster()
    
    # Find an interesting cluster (size > 2 and < 10 for good visualization)
    interesting_clusters = [
        c for c in clusters 
        if 3 <= len(c) <= 10
    ]
    
    if interesting_clusters:
        # Visualize first interesting cluster
        cluster_to_viz = interesting_clusters[0]
        
        print(f"\nVisualizing cluster with {len(cluster_to_viz)} mentions:")
        for mid in cluster_to_viz:
            print(f"  - {clusterer.mention_dict[mid]['mention']}")
        
        clusterer.visualize_cluster(
            cluster_to_viz,
            output_file='example_cluster.png'
        )
    else:
        print("No clusters of suitable size for visualization found.")


def example_with_evaluation(train_data, train_labels):
    """Example 5: With evaluation"""
    
    print("\n" + "="*70)
    print("EXAMPLE 5: GRAPH-BASED WITH EVALUATION")
    print("="*70)

    
    clusterer = GraphBasedClustering(
        mentions_data=train_data,
        similarity_threshold=0.80
    )
    
    # Measure time
    start_time = time.time()
    clusters = clusterer.cluster()
    runtime = time.time() - start_time
    
    # Evaluate
    f1_score = get_metrics(clusters, train_labels)
    
    print(f"\nResults:")
    print(f"  F1 Score: {f1_score:.3f}")
    print(f"  Runtime: {runtime:.2f}s")
    print(f"  Clusters: {len(clusters)}")
    print(f"  Throughput: {len(train_data)/runtime:.0f} mentions/second")


def example_noise_robustness(train_data, train_labels):
    """Example 6: Test robustness to noise"""
    
    print("\n" + "="*70)
    print("EXAMPLE 6: NOISE ROBUSTNESS")
    print("="*70)
    
    
    
    noise_levels = [0.0, 0.10, 0.20, 0.30, 0.40]
    
    for noise in noise_levels:
        print(f"\nNoise level: {noise:.0%}")
        
        # Inject noise
        if noise > 0:
            noisy_data = inject_boundary_errors(train_data, noise)
        else:
            noisy_data = train_data
        
        # Cluster
        clusterer = GraphBasedClustering(
            mentions_data=noisy_data,
            similarity_threshold=0.80
        )
        
        clusters = clusterer.cluster()
        
        # Evaluate
        f1 = get_metrics(clusters, train_labels)
        
        print(f"  F1: {f1:.3f}")
        print(f"  Clusters: {len(clusters)}")


# ============================================================================
# Main execution
# ============================================================================

# if __name__ == "__main__":
#     # Assuming you have train_data loaded
#     # train_data = [...]  # Your mention data
#     # train_labels = [...] # Your gold labels
    
#     # Run examples
    
#     # Example 1: Basic usage
#     clusters = example_basic_usage(train_data)
    
#     # Example 2: Compare thresholds
#     threshold_results = example_threshold_comparison(train_data)
    
#     # Example 3: Analyze transitivity
#     example_transitivity_analysis(train_data)
    
#     # Example 4: Visualize
#     example_visualization(train_data)
    
#     # Example 5: With evaluation
#     # example_with_evaluation(train_data, train_labels)
    
#     # Example 6: Noise robustness
#     # example_noise_robustness(train_data, train_labels)

In [5]:
# Initialize
clusterer = GraphBasedClustering(
    mentions_data = train_data,
    similarity_threshold=0.83
)

# Cluster
clusters = clusterer.cluster()


Building similarity graph...
  Nodes: 2974 mentions
  Similarity threshold: 0.83
  Computing 4,420,851 pairwise similarities...
  Added 84,756 edges in 29.63s
  Average degree: 57.00

Finding connected components...
  Found 779 clusters
  Cluster size distribution:
    Min: 1
    Max: 260
    Mean: 3.82
    Median: 1.0
    Singletons: 425


In [7]:
get_metrics(clusters, train_labels, verbose = False)

(np.float64(0.835835391316142),
 {'muc': {'precision': 0.9895216400911162,
   'recall': 0.9692101740294511,
   'f1': 0.9792605951307485},
  'b_cubed': {'precision': 0.9797831999902986,
   'recall': 0.8783729804146605,
   'f1': 0.9263108221055272},
  'ceafe': {'precision': np.float64(0.9241568934267055),
   'recall': np.float64(0.4463186088309055),
   'f1': np.float64(0.6019347567121504)},
  'lea': {'precision': 0.97785312634273,
   'recall': 0.8737176977440411,
   'f1': 0.9228570370257081}})

In [8]:
# Save predictions
def save_predictions(clusters, filename):
    
    with open(filename, 'w') as f:
        json.dump(clusters, f, indent = 2)
    
    print(f"prediction file saved.")

# Save results
save_predictions(clusters, '../../SOMD-2026/predictions/graph_based_clustering/graph-based-clusters.json')

prediction file saved.
